# Experiment A - forecast value under actuation delay (validation only)
Complete notebook: v5.2.1 pipeline cells (install, config, data, features, forecasting models, simulator, tuning) followed by the new experiment. Run all cells in order in ONE runtime (Runtime > Run all). The held-out test period is not used for any result here.
Output: `table_actuation_delay_validation.csv` (send it back) and the printed verdict per delay k.

### v5.2.1 pipeline - 0) install

In [ ]:
# 0) INSTALL (Colab)
!pip -q install pyarrow requests

### v5.2.1 pipeline - 1) imports + config

In [ ]:
# 1) IMPORTS + CONFIG
import os, sys, json, time, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_SOURCE  = "api"        # "api" = FREE public Bitcoin block API (no account, no key)  |  "bigquery" = needs a Google Cloud project
PROJECT_ID   = ""           # only used if DATA_SOURCE == "bigquery"  (your Google Cloud Project ID)
YEAR_START, YEAR_END = "2020-01-01", "2021-04-01"
MIN_BLOCKS   = 64000                      # 2020-01-01 -> 2021-04-01 has ~66,000 blocks
SEED         = 42
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20         # used ONLY in QUICK mode
T_TRAIN_END, T_VAL_END = "2020-10-01", "2021-01-01"   # train | validation (tuning) | CONFIRMATION TEST = 2021-01-01 .. 2021-03-31
EMBARGO      = 6                          # purge: demand label looks 6 blocks ahead
QUICK        = False
OUT_DIR      = "/content/results_v5_2_1"
os.makedirs(OUT_DIR, exist_ok=True)

# closed loop (walk-forward) on the DEMAND forecaster
CHUNK, WINDOW, PERIODIC_EVERY = 144, 20000, 20
PH_DELTA, PH_LAMBDA = 0.05, 1.0

# simulator (every number below is an explicit, testable assumption)
SIM = dict(
    CAP=100.0, CLASS_P=[0.10, 0.60, 0.30], CLASS_W=np.array([3.0, 1.0, 0.0]),   # HIGH, NORMAL, LOW
    T_AGE=1800.0, ANCHOR=1.0, OFFLOADABLE_P=0.5, MAX_WAIT_PARTIAL=1200.0,
    SIZE_SIGMA=0.8, SLA=3600.0, WARMUP_FRAC=0.05, DT_MEAN=600.0,
)
RHOS, N_SEEDS = [0.5, 0.7, 0.85, 0.95], 5
TUNE_RHOS, TUNE_SEEDS = [0.3, 0.6, 0.9], 1 # tuned on SEVERAL loads so the controller generalises (0.2 and 0.95 are unseen)
SLA_FLOOR = 0.90                           # pre-specified constraint: every class must keep SLA hit-rate >= 90 %
GRID_LO, GRID_HI = [0.0, 0.4, 0.8], [0.6, 1.0, 1.5, 2.5]
SENS_SEEDS = 2
EPS_TOL = 0.05          # eps-constraint: among configs within 5 % of the best latency objective pick the one using the LEAST off-chain
H_FWD = 6               # forecast horizon (blocks) = length of the demand window the forecaster predicts
RECAL_WINDOW, COOLDOWN = 6000, 2   # recalibration trains on the most recent blocks; min cycles between recalibrations
DRIFT_SCALE = 1.35      # drift-injection stress test: workload x1.35 from the middle of the test period
if QUICK:
    RHOS, N_SEEDS, TUNE_SEEDS, SENS_SEEDS, TUNE_RHOS = [0.9], 2, 1, 1, [0.3, 0.9]
    GRID_LO, GRID_HI = [0.0], [0.6, 1.5]
SIM["H"] = H_FWD
CLS = ["HIGH", "NORMAL", "LOW"]
import hashlib
CODE_VERSION = "v5.2.1"   # bump on ANY change that can alter results
NORMALISATION = "demand scale = TRAINING-period mean (fixed reference for validation and confirmation)"
CONFIG_HASH = hashlib.sha256(json.dumps(dict(CODE_VERSION=CODE_VERSION, NORMALISATION=NORMALISATION,
    SIM={k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in SIM.items()}, RHOS=RHOS, N_SEEDS=N_SEEDS, TUNE_RHOS=TUNE_RHOS,
    TUNE_SEEDS=TUNE_SEEDS, GRID_LO=GRID_LO, GRID_HI=GRID_HI, EPS_TOL=EPS_TOL, SLA_FLOOR=SLA_FLOOR, H_FWD=H_FWD, CHUNK=CHUNK,
    WINDOW=WINDOW, RECAL_WINDOW=RECAL_WINDOW, COOLDOWN=COOLDOWN, PH=(PH_DELTA, PH_LAMBDA), EMBARGO=EMBARGO, SEED=SEED,
    SPLIT=(YEAR_START, T_TRAIN_END, T_VAL_END, YEAR_END)), sort_keys=True).encode()).hexdigest()[:16]
print("Config OK | QUICK =", QUICK)
print("CODE VERSION:", CODE_VERSION, "|", NORMALISATION)
print("DESIGN FREEZE hash:", CONFIG_HASH, " <- record it; it must be identical in every run that you report")

### v5.2.1 pipeline - 2) data

In [ ]:
# 2) DATA EXTRACTION + COVERAGE GUARD  (this is what prevents the "15,000 blocks" mistake)
CACHE = f"{OUT_DIR}/blocks_{YEAR_START}_{YEAR_END}.parquet"

# ---------- FREE source: public Esplora-compatible block APIs (no account / no key / no billing) ----------
API_BASES = ["https://mempool.space/api", "https://blockstream.info/api"]
H_LO, H_HI = 610_000, 678_000         # generous height range that contains all of 2020; filtered by timestamp afterwards
if QUICK: H_HI = H_LO + 13_000

def fetch_from_api(workers=4):
    import requests, pickle, concurrent.futures as cf
    starts = list(range(H_HI, H_LO, -10))          # every request returns >= 10 blocks going DOWN from `start` -> full coverage
    ck = f"{OUT_DIR}/api_checkpoint.pkl"
    done = pickle.load(open(ck, "rb")) if os.path.exists(ck) else {}
    todo = [h for h in starts if h not in done]
    print(f"API fetch: {len(starts)} pages total, {len(done)} already cached, {len(todo)} to fetch (resumable)")
    sess = requests.Session(); sess.headers.update({"User-Agent": "research-notebook/1.0"})
    def one(h):
        for attempt in range(8):
            base = API_BASES[(attempt // 2) % len(API_BASES)]     # 2 tries per host, then switch host
            try:
                r = sess.get(f"{base}/blocks/{h}", timeout=30)
                if r.status_code == 200:
                    return h, [(b["height"], b["timestamp"], b["tx_count"], b["size"], b["weight"]) for b in r.json()]
            except Exception:
                pass
            time.sleep(min(30, 2 ** attempt))                     # back off on 429 / network errors
        return h, None
    failed, t0 = 0, time.time()
    with cf.ThreadPoolExecutor(max_workers=workers) as ex:
        for i, (h, rows) in enumerate(ex.map(one, todo), 1):
            if rows is None: failed += 1
            else: done[h] = rows
            if i % 250 == 0:
                pickle.dump(done, open(ck, "wb")); print(f"  {i}/{len(todo)} pages  ({time.time()-t0:.0f}s)")
    pickle.dump(done, open(ck, "wb"))
    if failed:
        raise RuntimeError(f"{failed} pages failed after retries. Just re-run this cell: it RESUMES from the checkpoint. "
                           "If it keeps failing, wait a few minutes (rate limit) or set DATA_SOURCE='bigquery'.")
    rows = [r for v in done.values() for r in v]
    r_ = pd.DataFrame(rows, columns=["number", "ts", "transaction_count", "size", "weight"]).drop_duplicates("number")
    r_["block_timestamp"] = pd.to_datetime(r_["ts"], unit="s", utc=True)
    r_ = r_[(r_.block_timestamp >= pd.Timestamp(YEAR_START, tz="UTC")) & (r_.block_timestamp < pd.Timestamp(YEAR_END, tz="UTC"))]
    r_ = r_.sort_values("number").reset_index(drop=True)[["number", "block_timestamp", "transaction_count", "size", "weight"]]
    r_.to_parquet(CACHE); return r_

# ---------- OPTIONAL source: BigQuery (free tier / sandbox, but needs a Google Cloud project you have access to) ----------
def resolve_project_id():
    pid = (PROJECT_ID or "").strip()
    bad = (not pid) or ("ENTER" in pid.upper()) or ("YOUR" in pid.upper()) or pid.startswith("<")
    if bad:
        try:
            import google.auth
            _, pid = google.auth.default()
        except Exception:
            pid = None
    if not pid:
        pid = input("Paste your Google Cloud PROJECT ID (not the project name): ").strip()
    print("Using BigQuery project:", pid); return pid

def fetch_from_bigquery():
    from google.colab import auth
    auth.authenticate_user()
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'google-cloud-bigquery', 'db-dtypes'])
    from google.cloud import bigquery
    client = bigquery.Client(project=resolve_project_id())
    q = f"""
    SELECT number, timestamp AS block_timestamp, transaction_count, size, weight
    FROM `bigquery-public-data.crypto_bitcoin.blocks`
    WHERE timestamp >= TIMESTAMP('{YEAR_START}') AND timestamp < TIMESTAMP('{YEAR_END}')
    ORDER BY number
    """
    try:
        r = client.query(q).to_dataframe()
    except Exception as e:
        raise RuntimeError("BigQuery failed (project id / permission / API not enabled). Easiest fix: set DATA_SOURCE='api'. Original: " + str(e))
    r.to_parquet(CACHE); return r

fetch = fetch_from_api if DATA_SOURCE == "api" else fetch_from_bigquery

def coverage_problems(r):
    msgs = []; ts = pd.to_datetime(r["block_timestamp"], utc=True)
    if len(r) < MIN_BLOCKS: msgs.append(f"only {len(r)} blocks (< {MIN_BLOCKS})")
    if ts.min() > pd.Timestamp(YEAR_START, tz="UTC") + pd.Timedelta(days=1): msgs.append(f"starts late: {ts.min()}")
    if ts.max() < pd.Timestamp(YEAR_END, tz="UTC") - pd.Timedelta(days=3): msgs.append(f"ends early: {ts.max()}")
    nn = np.sort(r["number"].values)
    if int(r["number"].duplicated().sum()): msgs.append("duplicate block numbers")
    if int((np.diff(nn) != 1).sum()): msgs.append("gaps in block numbers")
    return msgs

used_cache = os.path.exists(CACHE)
raw = pd.read_parquet(CACHE) if used_cache else fetch()
if not QUICK:
    probs = coverage_problems(raw)
    if probs and used_cache:
        print("STALE/PARTIAL CACHE ->", probs, "\nRe-fetching ...")
        os.remove(CACHE); raw = fetch(); probs = coverage_problems(raw)
    assert not probs, f"Data coverage check FAILED: {probs}"
    print("Coverage guard PASSED:", len(raw), "blocks,", raw.block_timestamp.min(), "->", raw.block_timestamp.max())
else:
    raw = raw.iloc[:12000].copy(); print("QUICK mode: coverage guard skipped")
display(raw.head())

### v5.2.1 pipeline - 3) features + split

In [ ]:
# 3) FEATURES (leak-free) + PURGED CHRONOLOGICAL SPLIT + DATASET AUDIT
# Unit = one Bitcoin block.  Row i uses ONLY information available after block i-1.
TARGET_INT = "block_time"   # inter-block timestamp interval (miner-reported)  -> Task A
TARGET_DEM = "demand"       # mean tx_count of blocks i..i+5 (workload intensity, forward window) -> Task B

def build_base_features(raw):
    d = raw.sort_values("number").reset_index(drop=True).copy()
    d["block_timestamp"] = pd.to_datetime(d["block_timestamp"], utc=True)
    d["dt_raw"] = d["block_timestamp"].diff().dt.total_seconds()
    n_neg = int((d["dt_raw"] < 0).sum())
    d[TARGET_INT] = d["dt_raw"].clip(lower=0)
    tx, sz, wt, bt = d["transaction_count"], d["size"], d["weight"], d[TARGET_INT]
    d[TARGET_DEM] = tx.iloc[::-1].rolling(6).mean().iloc[::-1]          # forward window i..i+5
    d["prev_tx"], d["prev_size"], d["prev_weight"] = tx.shift(1), sz.shift(1), wt.shift(1)
    d["hour"]      = d["block_timestamp"].shift(1).dt.hour
    d["dayofweek"] = d["block_timestamp"].shift(1).dt.dayofweek
    d["tx_ma_5"]     = tx.shift(1).rolling(5).mean()
    d["tx_std_5"]    = tx.shift(1).rolling(5).std()
    d["size_ma_5"]   = sz.shift(1).rolling(5).mean()
    d["weight_ma_5"] = wt.shift(1).rolling(5).mean()
    d["tx_ma_36"]    = tx.shift(1).rolling(36).mean()
    d["tx_ma_144"]   = tx.shift(1).rolling(144, min_periods=36).mean()
    d["prev_block_time"]       = bt.shift(1)
    d["prev_block_time_ma_3"]  = bt.shift(1).rolling(3).mean()
    d["prev_block_time_std_3"] = bt.shift(1).rolling(3).std()
    d["tx_pct_change"]     = tx.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    d["weight_pct_change"] = wt.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    return d, n_neg

base, n_neg_dt = build_base_features(raw)
BASE_COLS = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
             "tx_ma_36","tx_ma_144","prev_block_time","prev_block_time_ma_3","prev_block_time_std_3",
             "tx_pct_change","weight_pct_change"]
df = base.dropna(subset=BASE_COLS + [TARGET_INT, TARGET_DEM]).reset_index(drop=True)
ts_ = df["block_timestamp"]
if QUICK:
    n = len(df); i_tr = int(n * TRAIN_FRAC); i_va = int(n * (TRAIN_FRAC + VAL_FRAC))
else:                                              # DATE-BASED, out-of-time split
    i_tr = int((ts_ < pd.Timestamp(T_TRAIN_END, tz="UTC")).sum()); i_va = int((ts_ < pd.Timestamp(T_VAL_END, tz="UTC")).sum()); n = len(df)
    assert i_tr > 30000 and i_va - i_tr > 10000 and n - i_va > 10000, f"unexpected split sizes {i_tr}, {i_va - i_tr}, {n - i_va}"
a_tr, a_va = i_tr - EMBARGO, i_va - EMBARGO           # purged boundaries (train: [:a_tr], val: [i_tr:a_va], test: [i_va:])
tr = df.iloc[:a_tr]
NORM = dict(tx_mean=float(tr.prev_tx.mean()), tx_std=float(tr.prev_tx.std()),
            tx_max=float(tr.prev_tx.max()), w_max=float(tr.prev_weight.max()))
df["burst_flag"]     = (df["prev_tx"] > NORM["tx_mean"] + NORM["tx_std"]).astype(int)
df["queue_pressure"] = 0.5 * df["prev_tx"] / NORM["tx_max"] + 0.5 * df["prev_weight"] / NORM["w_max"]

FEATURES_INT = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
                "prev_block_time","prev_block_time_ma_3","prev_block_time_std_3","burst_flag","queue_pressure",
                "tx_pct_change","weight_pct_change"]
FEATURES_DEM = FEATURES_INT + ["tx_ma_36", "tx_ma_144"]
SIM["DT_MEAN"] = float(df[TARGET_INT].iloc[:a_tr].mean())
DEMAND_SCALE = float(df[TARGET_DEM].iloc[:a_tr].mean())   # TRAINING-only reference for normalising demand/forecasts (no val/test statistic enters the controller)

audit = dict(
    protocol="train 2020-01..09 | validation 2020-10..12 (all tuning) | held-out test 2021-01..03 (re-used across design iterations; not independent)", config_hash=CONFIG_HASH,
    observation_unit="one Bitcoin block", code_version=CODE_VERSION, demand_scale_train_mean=DEMAND_SCALE, data_source=DATA_SOURCE, data_source_detail=("public Esplora-compatible block APIs (mempool.space primary, blockstream.info fallback)" if DATA_SOURCE == "api" else "Google BigQuery bigquery-public-data.crypto_bitcoin.blocks"), block_height_min=int(raw["number"].min()), block_height_max=int(raw["number"].max()), duplicate_heights=int(raw["number"].duplicated().sum()), date_range=[YEAR_START, YEAR_END],
    blocks_fetched=int(len(raw)), rows_after_feature_construction=int(n), rows_dropped=int(len(raw) - n),
    n_features_interval_task=len(FEATURES_INT), n_features_demand_task=len(FEATURES_DEM),
    embargo_blocks=EMBARGO, n_train=int(a_tr), n_val=int(a_va - i_tr), n_test=int(n - i_va),
    train_range=[str(df.block_timestamp.iloc[0]), str(df.block_timestamp.iloc[a_tr-1])],
    val_range=[str(df.block_timestamp.iloc[i_tr]), str(df.block_timestamp.iloc[a_va-1])],
    test_range=[str(df.block_timestamp.iloc[i_va]), str(df.block_timestamp.iloc[-1])],
    negative_intervals_clipped_to_0=n_neg_dt,
    interval_mean_s=float(df[TARGET_INT].mean()), interval_cv=float(df[TARGET_INT].std() / df[TARGET_INT].mean()),
    interval_lag1_autocorr_train=float(df[TARGET_INT].iloc[:a_tr].autocorr(1)),
    demand_lag1_autocorr_train=float(df[TARGET_DEM].iloc[:a_tr].autocorr(1)),
    demand_mean_train_val_test=[float(df[TARGET_DEM].iloc[:a_tr].mean()), float(df[TARGET_DEM].iloc[i_tr:a_va].mean()),
                                float(df[TARGET_DEM].iloc[i_va:].mean())],
    norm_constants_from_train=NORM,
)
json.dump(audit, open(f"{OUT_DIR}/dataset_audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2))

### v5.2.1 pipeline - 4) forecasting models

In [ ]:
# 4) TWO PREDICTION TASKS (selection on VALIDATION, test touched once)
def make_models():
    ne = 100 if QUICK else 300
    wrap = lambda m: Pipeline([("sc", StandardScaler()), ("m", m)])
    return {
        "RandomForest": wrap(RandomForestRegressor(n_estimators=ne, max_depth=12, min_samples_split=5, random_state=SEED, n_jobs=-1)),
        "GradientBoosting": wrap(GradientBoostingRegressor(n_estimators=ne, learning_rate=0.03, max_depth=3, random_state=SEED)),
        "MLP": wrap(MLPRegressor(hidden_layer_sizes=(128, 64), activation="relu", solver="adam", learning_rate_init=0.0008,
                                 max_iter=700, early_stopping=True, random_state=SEED)),
        "Ridge": wrap(Ridge(alpha=1.0)),
    }
SL = dict(tr=slice(0, a_tr), va=slice(i_tr, a_va), te=slice(i_va, n))

def metrics(yt, yp):
    return dict(MAE=mean_absolute_error(yt, yp), RMSE=float(np.sqrt(mean_squared_error(yt, yp))), R2=r2_score(yt, yp))

def run_task(tag, target, feats, baselines, model_names):
    yy, Xf = df[target], df[feats]
    PV, PT, fitted, rows = {}, {}, {}, []
    for bn, fn in baselines.items():
        PV[bn], PT[bn] = fn(Xf.iloc[SL["va"]]), fn(Xf.iloc[SL["te"]])
    for nm in model_names:
        m = make_models()[nm]; t0 = time.perf_counter()
        m.fit(Xf.iloc[SL["tr"]], yy.iloc[SL["tr"]]); fitted[nm] = m
        PV[nm], PT[nm] = m.predict(Xf.iloc[SL["va"]]), m.predict(Xf.iloc[SL["te"]])
        print(f"[{tag}] {nm}: trained in {time.perf_counter() - t0:.1f}s")
    for nm in PV:
        mv, mt = metrics(yy.iloc[SL["va"]], PV[nm]), metrics(yy.iloc[SL["te"]], PT[nm])
        rows.append(dict(Model=nm, **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}))
    table = pd.DataFrame(rows).set_index("Model")
    sel = table.loc[model_names].sort_values("val_RMSE").index[0]
    best_base = table.loc[list(baselines)].sort_values("val_RMSE").index[0]
    return dict(tag=tag, target=target, feats=feats, table=table, SEL=sel, best_base=best_base, fitted=fitted,
                PV=PV, PT=PT, ref_mae=float(table.loc[sel, "val_MAE"]),
                y_va=yy.iloc[SL["va"]].values, y_te=yy.iloc[SL["te"]].values)

seas = df.iloc[:a_tr].groupby(df["hour"].iloc[:a_tr].astype(int))[TARGET_DEM].mean()
BASE_INT = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_INT].iloc[:a_tr].mean()),
            "Baseline: previous interval": lambda X: X["prev_block_time"].values}
BASE_DEM = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_DEM].iloc[:a_tr].mean()),
            "Baseline: persistence (5-block mean)": lambda X: X["tx_ma_5"].values,
            "Baseline: 36-block mean": lambda X: X["tx_ma_36"].values,
            "Baseline: hour-of-day mean": lambda X: X["hour"].astype(int).map(seas).fillna(seas.mean()).values}

TA = run_task("A: interval", TARGET_INT, FEATURES_INT, BASE_INT, ["RandomForest", "GradientBoosting", "MLP"])
TB = run_task("B: demand",   TARGET_DEM, FEATURES_DEM, BASE_DEM, ["Ridge", "RandomForest", "GradientBoosting", "MLP"])
for T in (TA, TB):
    print(f"\n===== Task {T['tag']}  (target = {T['target']}) =====")
    display(T["table"].round(4)); T["table"].to_csv(f"{OUT_DIR}/table_prediction_{T['tag'][0]}.csv")
    print("Selected on validation:", T["SEL"], "| best baseline on validation:", T["best_base"])

### v5.2.1 pipeline - 7) simulator

In [ ]:
# 7) TRACE-DRIVEN QUEUE SIMULATOR
# World (identical for every policy):
#  * block times      = REAL inter-block intervals of the period
#  * arrival process  = Poisson with intensity proportional to the REAL demand (forward 6-block mean tx_count),
#                       scaled so the average offered load = rho x block capacity (operator-calibrated mean rate)
#  * tx attributes    = class (HIGH/NORMAL/LOW), size (lognormal), off-loadable flag; PRE-GENERATED per (rho, seed)
# Decision timing: the controller chooses its regime at the START of each interval, seeing only the backlog left after the
# previous block + (for ADAPTIVE) a forecast of the demand of the coming interval.  ORACLE sees the true arrival count.
REG = {
    "STATIC":   dict(order="fifo", elig=(),     B=1),    # executed baseline: FIFO, full payload on-chain
    "FAST":     dict(order="fifo", elig=(),     B=1),
    "FIFO_OFF": dict(order="fifo", elig=(1, 2), B=10),   # static hybrid storage, no priority
    "PRIO":     dict(order="prio", elig=(),     B=1),    # priority+aging only
    "MID":      dict(order="prio", elig=(2,),   B=10),
    "CONG":     dict(order="prio", elig=(1, 2), B=25),
}
FIXED_CANDIDATES = ["FIFO_OFF", "PRIO", "MID", "CONG"]

def make_world(sl, rel_hat_map):
    dt = df[TARGET_INT].iloc[sl].values.astype(float); dem = df[TARGET_DEM].iloc[sl].values.astype(float); sc = DEMAND_SCALE   # fixed TRAINING mean: no information from the evaluated period
    rh = {k: np.clip(np.asarray(v) / sc, 0.2, 3.0) for k, v in rel_hat_map.items()}
    rh["ADAPTIVE_CONST"] = np.ones(len(dt))          # ablation: knows only the AVERAGE load level, no time-varying forecast
    return dict(dt=dt, rel=dem / sc, rel_hat=rh)

def make_arrivals(world, rho, seed, S):
    rng = np.random.default_rng(seed); dt = world["dt"]
    n_arr = rng.poisson(rho * S["CAP"] / 600.0 * world["rel"] * dt); N = int(n_arr.sum())
    t_end = np.cumsum(dt); t_start = t_end - dt; blk = np.repeat(np.arange(len(dt)), n_arr)
    times = np.sort(t_start[blk] + rng.random(N) * dt[blk]); sg = S["SIZE_SIGMA"]
    return dict(t=times, cls=rng.choice(3, N, p=S["CLASS_P"]), size=rng.lognormal(-sg**2 / 2, sg, N),
                off=rng.random(N) < S["OFFLOADABLE_P"], offs=np.concatenate([[0], np.cumsum(n_arr)]), t_end=t_end)

def select_block(t, q_t, q_c, q_s, q_o, reg, S):
    nq = len(q_t)
    if nq == 0: return np.zeros(0, bool), 0.0
    if reg["order"] == "fifo": o = np.arange(nq)
    else: o = np.argsort(-(S["CLASS_W"][q_c] + (t - q_t) / S["T_AGE"]), kind="stable")
    elig = (np.isin(q_c, reg["elig"]) & q_o) if reg["elig"] else np.zeros(nq, bool)
    el_o, pos, so, to = elig[o], np.arange(nq), q_s[o], q_t[o]
    ne_pos, ne_cost = pos[~el_o], so[~el_o]; el_pos = pos[el_o]; m = len(el_pos); B = reg["B"]
    n_full = m // B; rem = m - n_full * B; nb = n_full
    if rem > 0 and (t - to[el_pos[n_full * B:]]).max() > S["MAX_WAIT_PARTIAL"]: nb += 1
    first = el_pos[::B][:nb] if nb > 0 else np.zeros(0, int)
    ranks = np.concatenate([ne_pos, first]); costs = np.concatenate([ne_cost, np.full(nb, S["ANCHOR"])])
    oi = np.argsort(ranks, kind="stable"); cum = np.cumsum(costs[oi]); k = int(np.searchsorted(cum, S["CAP"], side="right"))
    if k == 0: return np.zeros(nq, bool), 0.0
    inc = oi[:k]; is_ne = inc < len(ne_pos); inc_o = np.zeros(nq, bool); inc_o[ne_pos[inc[is_ne]]] = True
    if nb > 0:
        inc_b = np.zeros(nb, bool); inc_b[inc[~is_ne] - len(ne_pos)] = True
        bid = np.arange(m) // B; ok = bid < nb; inc_o[el_pos[ok][inc_b[bid[ok]]]] = True
    mask = np.zeros(nq, bool); mask[o[inc_o]] = True
    return mask, float(cum[k - 1])

def decide(policy, j, br0, arr, world, rho, S, P):
    """Regime chosen at the START of interval j (priority ordering is ALWAYS on; only the off-chain batching level adapts):
       s <= lo -> PRIO (no batching) | lo < s < hi -> MID | s >= hi -> CONG.
       FEEDBACK: s = backlog (in block capacities).  ADAPTIVE: backlog + forecast load.  ORACLE: backlog + TRUE future arrivals.
       Statistic mode 'I' = coming interval (s = backlog + load);  'H' = predicted backlog growth over the next H blocks."""
    if policy == "STATIC":     return "STATIC"
    if policy == "PRIO_ONLY":  return "PRIO"
    if policy == "BEST_FIXED": return P["BEST_FIXED"]
    if policy.startswith("FIXED:"): return policy.split(":")[1]
    H = S["H"]
    if policy == "FEEDBACK":
        s, (lo, hi) = br0, P["FEEDBACK"]
    else:
        key = policy if policy in ("ORACLE", "ADAPTIVE_CONST") else "ADAPTIVE"; lo, hi, mode = P[key]
        if policy == "ORACLE":
            nb_ = len(arr["offs"]) - 1
            if mode == "I": s = br0 + (arr["offs"][j + 1] - arr["offs"][j]) / S["CAP"]
            else: je = min(j + H, nb_); s = max(0.0, br0 + (arr["offs"][je] - arr["offs"][j]) / S["CAP"] - (je - j))
        else:
            u_hat = rho * world["rel_hat"][policy][j] * S["DT_MEAN"] / 600.0
            s = br0 + u_hat if mode == "I" else max(0.0, br0 + H * (u_hat - 1.0))
    return "CONG" if s >= hi else ("PRIO" if s <= lo else "MID")

def simulate(arr, world, policy, rho, S, P):
    nb = len(world["dt"]); offs = arr["offs"]
    q_t, q_c, q_s, q_o = np.empty(0), np.empty(0, int), np.empty(0), np.empty(0, bool)
    L, C, A, CT, used, backlog, counts = [], [], [], [], np.zeros(nb), np.zeros(nb), {}
    for j in range(nb):
        rname = decide(policy, j, len(q_t) / S["CAP"], arr, world, rho, S, P); counts[rname] = counts.get(rname, 0) + 1
        s, e = offs[j], offs[j + 1]
        if e > s:
            q_t = np.concatenate([q_t, arr["t"][s:e]]); q_c = np.concatenate([q_c, arr["cls"][s:e]])
            q_s = np.concatenate([q_s, arr["size"][s:e]]); q_o = np.concatenate([q_o, arr["off"][s:e]])
        t = arr["t_end"][j]; mask, u = select_block(t, q_t, q_c, q_s, q_o, REG[rname], S)
        if mask.any():
            L.append(t - q_t[mask]); C.append(q_c[mask]); A.append(q_t[mask]); CT.append(np.full(int(mask.sum()), t)); keep = ~mask
            q_t, q_c, q_s, q_o = q_t[keep], q_c[keep], q_s[keep], q_o[keep]
        used[j] = u; backlog[j] = len(q_t)
    cat = lambda l, d: np.concatenate(l) if l else np.empty(0, d)
    return dict(lat=cat(L, float), cls=cat(C, int), arr_t=cat(A, float), comp_t=cat(CT, float), arrivals_t=arr["t"], arrivals_cls=arr["cls"], used=used, backlog=backlog, q_t=q_t, q_c=q_c,
                t_end=arr["t_end"][-1], t_warm=arr["t_end"][int(S["WARMUP_FRAC"] * nb)], j_warm=int(S["WARMUP_FRAC"] * nb),
                n_arrived=len(arr["t"]), reg=counts)

def summarize(res, S, seg=0):
    """seg=0 -> whole period after warm-up; seg=1..4 -> consecutive quarters of it.
    Percentiles INCLUDE still-unfinished tx (their current wait = lower bound) so starvation cannot hide."""
    edges = np.linspace(res["t_warm"], res["t_end"], 5)
    lo, hi = (res["t_warm"], np.inf) if seg == 0 else (edges[seg - 1], edges[seg] if seg < 4 else np.inf)
    m = (res["arr_t"] >= lo) & (res["arr_t"] < hi); lat, cl = res["lat"][m], res["cls"][m]
    um = (res["q_t"] >= lo) & (res["q_t"] < hi); ua, uc = res["t_end"] - res["q_t"][um], res["q_c"][um]
    rows = []
    for name, sel in [("ALL", None), ("HIGH", 0), ("NORMAL", 1), ("LOW", 2)]:
        l = lat if sel is None else lat[cl == sel]; a = ua if sel is None else ua[uc == sel]; both = np.concatenate([l, a])
        pc = np.percentile(both, [50, 95, 99]) if len(both) else [np.nan] * 3
        rows.append(dict(cls=name, n_done=len(l), n_unfinished=len(a), mean=both.mean() if len(both) else np.nan,
                         std=both.std() if len(both) else np.nan, p50=pc[0], p95=pc[1], p99=pc[2],
                         max_wait=both.max() if len(both) else np.nan, sla_hit=(l <= S["SLA"]).sum() / max(len(both), 1)))
    bl = res["backlog"][res["j_warm"]:]; tot = sum(res["reg"].values())
    t_hi = res["t_end"] if not np.isfinite(hi) else hi; dur = max(t_hi - lo, 1.0)           # window [lo, t_hi)
    done_w = (res["comp_t"] >= lo) & (res["comp_t"] < t_hi) if np.isfinite(hi) else (res["comp_t"] >= lo)
    arr_w = (res["arrivals_t"] >= lo) & (res["arrivals_t"] < t_hi) if np.isfinite(hi) else (res["arrivals_t"] >= lo)
    comp_cls = res["cls"][done_w]; arr_cls = res["arrivals_cls"][arr_w]
    glob = dict(completed_tps=done_w.sum() / dur,                 # confirmations per second INSIDE the window (true completed throughput)
                arrival_tps=arr_w.sum() / dur,                    # offered arrival rate in the same window
                completed_to_arrived_ratio=done_w.sum() / max(arr_w.sum(), 1),
                completed_tps_HIGH=(comp_cls == 0).sum() / dur, completed_tps_NORMAL=(comp_cls == 1).sum() / dur, completed_tps_LOW=(comp_cls == 2).sum() / dur,
                arrival_tps_HIGH=(arr_cls == 0).sum() / dur, arrival_tps_NORMAL=(arr_cls == 1).sum() / dur, arrival_tps_LOW=(arr_cls == 2).sum() / dur,
                onchain_units_per_tx=res["used"].sum() / max(len(res["lat"]), 1), backlog_mean=bl.mean(),
                backlog_p95=np.percentile(bl, 95), backlog_max=bl.max(), backlog_final=res["backlog"][-1],
                frac_time_overloaded=float((bl > 5 * S["CAP"]).mean()),
                **{f"share_{k}": res["reg"].get(k, 0) / tot for k in ["STATIC", "FAST", "FIFO_OFF", "PRIO", "MID", "CONG"]})
    return rows, glob
print("Simulator ready.")

### v5.2.1 pipeline - 8) tuning on validation

In [ ]:
# 8) TUNE THE CONTROLLERS ON THE VALIDATION PERIOD ONLY (no test-set tuning; every policy gets the same tuning budget)
# Pre-specified two-stage objective (epsilon-constraint):
#   latency objective  L = p95(HIGH) + mean-class p95   (+ 1e5 x violation of "every class SLA hit-rate >= SLA_FLOOR")
#   stage 1: L* = best latency objective reachable by ANY non-oracle policy/configuration
#   stage 2: each policy picks, among its configurations with L <= (1+EPS_TOL) x L*, the one with the LEAST off-chain use.
# => adaptation is rewarded for keeping payloads ON-CHAIN whenever the service level allows it.
dem_val_pred = TB["fitted"][TB["SEL"]].predict(df[FEATURES_DEM].iloc[SL["va"]])
W_VAL = make_world(SL["va"], {"ADAPTIVE": dem_val_pred})
ARR_VAL = {(rho, s_): make_arrivals(W_VAL, rho, 1000 + 17 * i + s_, SIM) for i, rho in enumerate(TUNE_RHOS) for s_ in range(TUNE_SEEDS)}

def objective_from_rows(rows):
    p = {r["cls"]: r for r in rows}; sla_min = min(p[c]["sla_hit"] for c in CLS)
    return p["HIGH"]["p95"] + np.mean([p[c]["p95"] for c in CLS]) + 1e5 * max(0.0, SLA_FLOOR - sla_min)

def tune_eval(policy, P):
    objs, offl = [], []
    for (rho, _), arr in ARR_VAL.items():
        rows, g = summarize(simulate(arr, W_VAL, policy, rho, SIM, P), SIM); objs.append(objective_from_rows(rows)); offl.append(1 - g["onchain_units_per_tx"])
    return float(np.mean(objs)), float(np.mean(offl))

grid = [(lo, hi) for lo in GRID_LO for hi in GRID_HI if hi > lo]
cands = []
st_obj, st_off = tune_eval("STATIC", {}); cands.append(dict(key="STATIC", params="-", obj=st_obj, offload=st_off))
for r in FIXED_CANDIDATES:
    o, f = tune_eval(f"FIXED:{r}", {}); cands.append(dict(key="BEST_FIXED", params=r, obj=o, offload=f))
for g in grid:
    o, f = tune_eval("FEEDBACK", {"FEEDBACK": g}); cands.append(dict(key="FEEDBACK", params=g, obj=o, offload=f))
for pol in ["ADAPTIVE", "ADAPTIVE_CONST", "ORACLE"]:
    for mode in ["I", "H"]:                      # statistic mode is a validation-tuned hyper-parameter
        for g in grid:
            p3 = (g[0], g[1], mode); o, f = tune_eval(pol, {pol: p3}); cands.append(dict(key=pol, params=p3, obj=o, offload=f))
tune_df = pd.DataFrame(cands); tune_df["params"] = tune_df["params"].astype(str); tune_df.to_csv(f"{OUT_DIR}/table_tuning_validation.csv", index=False)
L_star = tune_df[tune_df.key.isin(["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST"])].obj.min()
TUNED = {}
for key in ["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST", "ORACLE"]:
    pool = [c for c in cands if c["key"] == key]; ok = [c for c in pool if c["obj"] <= (1 + EPS_TOL) * L_star]
    best = min(ok, key=lambda c: c["offload"]) if ok else min(pool, key=lambda c: c["obj"])
    TUNED[key] = best["params"]; print(f" {key:15s} -> {best['params']!s:12s} latency-obj={best['obj']:9.0f}  off-chain share={best['offload']:.3f}  ({'within eps' if ok else 'NO config within eps: using best latency'})")
print(f"\nBest reachable latency objective L* = {L_star:.0f}; STATIC objective = {st_obj:,.0f} (>1e4 means SLA violated on validation)")

### NEW - Experiment A: actuation delay

In [ ]:
# =====================================================================================================
# EXPERIMENT A (validation period ONLY): does the demand forecast help once regime changes take effect k blocks late?
# Paste this as ONE new cell at the END of the v5.2.1 notebook (after the tuning cell has run, i.e. after TUNED/L_star exist).
# It does not modify any existing variable or result.  Nothing here touches the held-out test period.
#
# Pre-specified question and decision rule (written BEFORE seeing results):
#   Delay model : a regime chosen at the start of block j is applied at block j+k (k = 0, 2, 5, 10); before that the previous
#                 decision stays in force.  (Fabric-like batch timeout / anchoring setup time.)  k = 0 is the main-paper setting.
#   Policies    : BEST_FIXED (CONG), FEEDBACK (backlog only), ADAPTIVE_CONST (backlog + average load), ADAPTIVE (backlog + forecast).
#   Tuning      : per k, on validation, same grid/budget/eps-rule as the paper (3 loads, 1 seed).
#   Evaluation  : 20 fresh validation seeds (5000+), loads 0.5/0.7/0.85/0.95, identical arrivals for all policies, paired 95% CIs.
#   "Forecast helps at k" iff  (saving_ADAPTIVE - saving_ADAPTIVE_CONST) has a 95% CI above 0  AND
#                              (latency_ADAPTIVE - latency_ADAPTIVE_CONST) <= +0.5 percentage points.
# =====================================================================================================
import numpy as np, pandas as pd, time, itertools, scipy.stats as st

K_LIST   = [0, 2, 5, 10]                 # actuation delays (blocks) to test
N_EVAL   = 20                            # evaluation seeds (reduce to 10 for a quicker first look)
EVAL_RHO = [0.5, 0.7, 0.85, 0.95]
GRID_LO_ = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]; GRID_HI_ = [0.4, 0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]   # fine grid, same for every policy
# (a coarse grid makes ADAPTIVE_CONST degenerate: it then sits in CONG all the time and saves nothing, which would flatter the forecast)
POLS     = ["FEEDBACK", "ADAPTIVE_CONST", "ADAPTIVE"]

def simulate_delay(arr, world, policy, rho, S, P, k):
    nb = len(world["dt"]); offs = arr["offs"]
    q_t, q_c, q_s, q_o = np.empty(0), np.empty(0, int), np.empty(0), np.empty(0, bool)
    L, C, A, CT, used, backlog, counts, decided = [], [], [], [], np.zeros(nb), np.zeros(nb), {}, []
    for j in range(nb):
        decided.append(decide(policy, j, len(q_t) / S["CAP"], arr, world, rho, S, P))      # decision made now ...
        rname = decided[j - k] if j >= k else "CONG"                                          # ... takes effect k blocks later
        counts[rname] = counts.get(rname, 0) + 1
        s, e = offs[j], offs[j + 1]
        if e > s:
            q_t = np.concatenate([q_t, arr["t"][s:e]]); q_c = np.concatenate([q_c, arr["cls"][s:e]])
            q_s = np.concatenate([q_s, arr["size"][s:e]]); q_o = np.concatenate([q_o, arr["off"][s:e]])
        t = arr["t_end"][j]; mask, u = select_block(t, q_t, q_c, q_s, q_o, REG[rname], S)
        if mask.any():
            L.append(t - q_t[mask]); C.append(q_c[mask]); A.append(q_t[mask]); CT.append(np.full(int(mask.sum()), t)); keep = ~mask
            q_t, q_c, q_s, q_o = q_t[keep], q_c[keep], q_s[keep], q_o[keep]
        used[j] = u; backlog[j] = len(q_t)
    cat = lambda l, d: np.concatenate(l) if l else np.empty(0, d)
    return dict(lat=cat(L, float), cls=cat(C, int), arr_t=cat(A, float), comp_t=cat(CT, float), arrivals_t=arr["t"], arrivals_cls=arr["cls"], used=used, backlog=backlog, q_t=q_t, q_c=q_c,
                t_end=arr["t_end"][-1], t_warm=arr["t_end"][int(S["WARMUP_FRAC"] * nb)], j_warm=int(S["WARMUP_FRAC"] * nb), n_arrived=len(arr["t"]), reg=counts)

def run_one(arr, pol, rho, P, k):
    res = simulate_delay(arr, W_D, pol, rho, SIM, P, k); rows, g = summarize(res, SIM)
    p = {r["cls"]: r for r in rows}
    macro = float(np.mean([p[c]["p95"] for c in CLS]))
    return dict(obj=objective_from_rows(rows), macro=macro, off=1 - g["onchain_units_per_tx"], sla_min=min(p[c]["sla_hit"] for c in CLS))

# ---- world on the validation period (forecast = the paper's selected model; training-only demand scale) ----
dem_val_pred_D = TB["fitted"][TB["SEL"]].predict(df[FEATURES_DEM].iloc[SL["va"]])
W_D = make_world(SL["va"], {"ADAPTIVE": dem_val_pred_D})

# ---- sanity: k = 0 must reproduce the paper's simulator exactly ----
_a = make_arrivals(W_D, 0.7, 4242, SIM); _P = {"ADAPTIVE": (0.4, 0.6, "I"), "FEEDBACK": (0.4, 0.6), "ADAPTIVE_CONST": (0.4, 0.6, "I"), "BEST_FIXED": "CONG"}
for _pol in POLS:
    _r0 = simulate(_a, W_D, _pol, 0.7, SIM, _P); _r1 = simulate_delay(_a, W_D, _pol, 0.7, SIM, _P, 0)
    assert np.array_equal(_r0["lat"], _r1["lat"]) and np.array_equal(_r0["backlog"], _r1["backlog"]), f"k=0 mismatch for {_pol}"
print("sanity OK: k=0 reproduces the paper's simulator for", POLS)

ARR_T = {(rho, i): make_arrivals(W_D, rho, 1000 + 17 * i, SIM) for i, rho in enumerate(TUNE_RHOS)}        # tuning arrivals (1 seed)
ARR_E = {(rho, s): make_arrivals(W_D, rho, 5000 + 31 * s + int(rho * 100), SIM) for rho in EVAL_RHO for s in range(N_EVAL)}   # evaluation arrivals

def tune_k(k):
    def ev(pol, P):
        r = [run_one(a, pol, rho, P, k) for (rho, _), a in ARR_T.items()]
        return float(np.mean([x["obj"] for x in r])), float(np.mean([x["off"] for x in r]))
    pairs = [(lo, hi) for lo in GRID_LO_ for hi in GRID_HI_ if hi > lo]; cands = []
    o, f = ev("BEST_FIXED", {"BEST_FIXED": "CONG"}); cands.append(dict(key="BEST_FIXED", params="CONG", obj=o, offload=f))
    for g in pairs:
        o, f = ev("FEEDBACK", {"FEEDBACK": g}); cands.append(dict(key="FEEDBACK", params=g, obj=o, offload=f))
    for pol in ["ADAPTIVE_CONST", "ADAPTIVE"]:
        for mode in ["I"]:
            for g in pairs:
                p3 = (g[0], g[1], mode); o, f = ev(pol, {pol: p3}); cands.append(dict(key=pol, params=p3, obj=o, offload=f))
    Ls = min(c["obj"] for c in cands); T = {}
    for key in ["BEST_FIXED"] + POLS:
        pool = [c for c in cands if c["key"] == key]; ok = [c for c in pool if c["obj"] <= (1 + EPS_TOL) * Ls]
        T[key] = (min(ok, key=lambda c: c["offload"]) if ok else min(pool, key=lambda c: c["obj"]))["params"]
    return T

def ci(x):
    x = np.asarray(x, float); n = len(x); h = st.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return x.mean(), h

rows_out, t0 = [], time.time()
for k in K_LIST:
    T = tune_k(k); print(f"\n--- k={k}: tuned ({time.time()-t0:.0f}s) ->", T, flush=True)
    P = dict(BEST_FIXED="CONG", FEEDBACK=T["FEEDBACK"], ADAPTIVE_CONST=T["ADAPTIVE_CONST"], ADAPTIVE=T["ADAPTIVE"])
    per = {pol: {s: [] for s in range(N_EVAL)} for pol in ["BEST_FIXED"] + POLS}
    for (rho, s), a in ARR_E.items():
        for pol in ["BEST_FIXED"] + POLS:
            per[pol][s].append(run_one(a, pol, rho, P, k))
    M = {pol: {m: np.array([np.mean([r[m] for r in per[pol][s]]) for s in range(N_EVAL)]) for m in ("macro", "off", "sla_min")} for pol in per}
    for pol in POLS:
        sav = M["BEST_FIXED"]["off"] - M[pol]["off"]; lat = 100 * (M[pol]["macro"] / M["BEST_FIXED"]["macro"] - 1)
        rows_out.append(dict(k=k, policy=pol, params=str(T[pol]), saving=sav.mean(), saving_ci=ci(sav)[1], latency_cost_pct=lat.mean(), latency_ci=ci(lat)[1], min_sla=M[pol]["sla_min"].mean()))
    sa, sc_, sf = (M["BEST_FIXED"]["off"] - M[p]["off"] for p in ("ADAPTIVE", "ADAPTIVE_CONST", "FEEDBACK"))
    la, lc, lf = (100 * (M[p]["macro"] / M["BEST_FIXED"]["macro"] - 1) for p in ("ADAPTIVE", "ADAPTIVE_CONST", "FEEDBACK"))
    d_s, h_s = ci(sa - sc_); d_l, h_l = ci(la - lc)
    helps = (d_s - h_s > 0) and (d_l <= 0.5)
    if sc_.mean() < 0.02: print("   WARNING: ADAPTIVE_CONST saves almost nothing here (degenerate comparison); do not read the verdict below.")
    print(f"k={k:2d}  ADAPTIVE - CONST : saving {d_s:+.4f} ± {h_s:.4f}   latency {d_l:+.2f} ± {h_l:.2f} pp   => forecast helps? {helps}")
    ds2, hs2 = ci(sa - sf); dl2, hl2 = ci(la - lf)
    print(f"        ADAPTIVE - FEEDBACK: saving {ds2:+.4f} ± {hs2:.4f}   latency {dl2:+.2f} ± {hl2:.2f} pp")
    rows_out.append(dict(k=k, policy="DIFF ADAPTIVE-CONST", saving=d_s, saving_ci=h_s, latency_cost_pct=d_l, latency_ci=h_l, params=f"forecast_helps={helps}"))
    rows_out.append(dict(k=k, policy="DIFF ADAPTIVE-FEEDBACK", saving=ds2, saving_ci=hs2, latency_cost_pct=dl2, latency_ci=hl2, params=""))
R = pd.DataFrame(rows_out); R.to_csv("table_actuation_delay_validation.csv", index=False)
print("\n", R.round(4).to_string(index=False)); print(f"\nTotal time {time.time()-t0:.0f}s.  Saved table_actuation_delay_validation.csv  (send me this CSV)")
